# Phiên song song B2 – train seed 1234 và 2025 cho TVPReid-iLIDs

Notebook này chỉ train **TVPReid-iLIDs** (2 lần, khoảng 5–6 giờ) để chạy song song với `paper.ipynb`.
Mã nguồn, dữ liệu và kết quả dùng chung thư mục `pbl6remake/srccv` trên Drive.

**Quy tắc để không ghi đè lên phiên khác**
- Chạy lần lượt các ô dưới đây (0.2 → 0.3 → 0.4 → 0.5 → train → đánh giá). Ô 0.4 chặn việc train bằng `train.py` cũ.
- Không chạy TVPReid-iLIDs ở phiên `paper.ipynb` (ô B.2 trong đó đã được gỡ khỏi hàng đợi).
- Mất kết nối: chạy lại từ ô đầu tiên; lần train dở sẽ tiếp tục từ epoch gần nhất.
- Xong ô cuối, quay lại `paper.ipynb` chạy B.4 → E.1 → E.2 để tổng hợp và điền bài.
- Trong log train, cột `Mot:` phải giảm dưới khoảng 2,5 sau vài epoch. Nếu đứng yên ở 2,77 (= ln 16) thì nhánh chuyển động không học – dừng lại và kiểm tra mã.

### 0.2 Cấu hình

In [ ]:
# ===== 0.2 CẤU HÌNH (sửa nếu thư mục của bạn khác) =====
DRIVE_PROJECT_ROOT = "/content/drive/MyDrive/pbl6remake"      # thư mục dự án trên Drive
SRC_DIR   = DRIVE_PROJECT_ROOT + "/srccv"                     # mã nguồn (chứa train.py)
CODE_ZIP  = DRIVE_PROJECT_ROOT + "/revision_code.zip"         # gói mã bổ sung tải lên từ máy
DATA_ROOT = "/content/dataset"                                # dữ liệu giải nén ra ổ SSD của Colab (nhanh hơn Drive)
DATASET_SOURCES = ["TVPReid_dataset.zip", "dataset.zip", "TVPReid_dataset"]   # tìm lần lượt trong DRIVE_PROJECT_ROOT

SUBSETS   = ["TVPReid-PRID", "TVPReid-iLIDs", "TVPReid-Duke"]
SEEDS_NEW = [1234, 2025]                                      # Phần B: seed mới (seed 42 dùng lại checkpoint cũ)
LOO_RUNS  = ["no_init", "no_temporal", "no_dsca", "no_mim"]   # Phần C: ablation độc lập (bắt buộc)
CUM_RUNS  = ["cum1", "cum2", "cum3"]                          # Phần D: ablation cộng dồn (khuyến nghị)
print("OK – cấu hình đã nạp")

### 0.3 Kết nối Drive, GPU, thư viện (chọn đúng tài khoản có thư mục pbl6remake)

In [ ]:
import os, sys, json, time, shutil, zipfile, codecs, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

def run(cmd, check=True):
    # Chạy một lệnh shell trong SRC_DIR và in kết quả ngay khi có (kể cả thanh tiến trình của train.py).
    print("$", cmd, flush=True)
    env = dict(os.environ, PYTHONUNBUFFERED="1")
    p = subprocess.Popen(cmd, shell=True, cwd=SRC_DIR, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    dec = codecs.getincrementaldecoder("utf-8")("replace")
    while True:
        chunk = p.stdout.read1(65536)
        if not chunk:
            break
        sys.stdout.write(dec.decode(chunk))
        sys.stdout.flush()
    rc = p.wait()
    if check and rc != 0:
        raise RuntimeError(f"Lệnh thất bại (mã {rc}) – đọc thông báo lỗi ở trên, xem Phụ lục 'Xử lý sự cố'.")
    return rc

def rr(stage, *extra):
    # Gọi experiments/run_revision.py cho một bước (stage). Thêm "--dry_run" để chỉ in lệnh.
    return run(f"python experiments/run_revision.py --stage {stage} --data_root {DATA_ROOT} " + " ".join(extra))

def seeds_to_train(sub):
    # Seed mới của Phần B; thêm seed 42 nếu không còn checkpoint của lần nộp trước cho tập con này.
    has_42 = Path(SRC_DIR, "checkpoints", sub, "best.pth").exists() or \
             Path(SRC_DIR, "checkpoints", sub, "seed42", "best.pth").exists()
    return "--seeds " + " ".join(map(str, SEEDS_NEW + ([] if has_42 else [42])))

import torch
if torch.cuda.is_available():
    prop = torch.cuda.get_device_properties(0)
    print(f"✅ GPU: {prop.name}, {prop.total_memory / 1024**3:.1f} GB")
    if prop.total_memory < 14 * 1024**3:
        print("⚠️ GPU dưới 14 GB: train với lô 16 có thể hết bộ nhớ – nên đổi sang T4/L4/A100.")
else:
    print("⚠️ CHƯA CÓ GPU: Runtime → Change runtime type → GPU. Phần A.4 và mọi bước train đều cần GPU.")

if not os.path.isdir(SRC_DIR):
    raise FileNotFoundError(f"Không thấy {SRC_DIR}. Kiểm tra lại ô 0.2 hoặc Drive đã kết nối chưa.")
if IN_COLAB:
    run("pip install -q transformers pyyaml sentencepiece sacremoses")
print("OK")

### 0.4 Đồng bộ mã nguồn (bắt buộc)
So từng file trong `revision_code.zip` với `srccv/` trên Drive; file nào thiếu hoặc khác thì được thay (bản cũ sao lưu vào `_backup_before_revision/`). Mã cũ có thể bỏ qua `--seed/--tag` và **ghi đè `best.pth` của seed 42**, hoặc dùng cấu hình khác với lần train gốc.

In [ ]:
def stale_files():
    # Files in revision_code.zip whose copy in srccv/ is missing or different (old train.py, configs, data/, loss/ ...).
    with zipfile.ZipFile(CODE_ZIP) as z:
        return [n for n in z.namelist() if not n.endswith("/")
                and (not Path(SRC_DIR, n).exists() or Path(SRC_DIR, n).read_bytes() != z.read(n))]

if not os.path.exists(CODE_ZIP):
    raise FileNotFoundError(f"Không thấy {CODE_ZIP} – tải revision_code.zip mới nhất lên Drive trước.")
stale = stale_files()
if stale:
    backup = Path(SRC_DIR) / "_backup_before_revision" / time.strftime("%Y%m%d_%H%M%S")
    with zipfile.ZipFile(CODE_ZIP) as z:
        for n in stale:
            old = Path(SRC_DIR) / n
            if old.exists():
                (backup / n).parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(old, backup / n)
            z.extract(n, SRC_DIR)
    for d in Path(SRC_DIR).rglob("__pycache__"):
        shutil.rmtree(d, ignore_errors=True)
    print(f"Đã cập nhật {len(stale)} file khác với revision_code.zip: {', '.join(stale)}")
    print(f"Bản cũ lưu ở {backup}")
assert not stale_files(), "Mã trên Drive vẫn khác revision_code.zip – KHÔNG train"
print("✅ Mã nguồn trên Drive khớp revision_code.zip (train.py, configs, data, loss, models, utils)")


### 0.5 Giải nén dữ liệu (vài phút)

In [ ]:
import pandas as pd

def caption_dirs(root):
    return sorted(p.parent for p in Path(root).rglob("captions") if p.is_dir()) if os.path.isdir(root) else []

os.makedirs(DATA_ROOT, exist_ok=True)
if not caption_dirs(DATA_ROOT):
    for name in DATASET_SOURCES:
        src = os.path.join(DRIVE_PROJECT_ROOT, name)
        if name.endswith(".zip") and os.path.isfile(src):
            print(f"Giải nén {src} → {DATA_ROOT} ...")
            with zipfile.ZipFile(src) as z:
                z.extractall(DATA_ROOT)
            break
        if os.path.isdir(src):
            print(f"Sao chép {src} → {DATA_ROOT} ...")
            shutil.copytree(src, os.path.join(DATA_ROOT, name), dirs_exist_ok=True)
            break

EXPECTED_SPLIT = {"PRID": (921, 71, 142), "ILIDS": (488, 37, 75), "DUKE": (3920, 302, 603)}
found = caption_dirs(DATA_ROOT)
if not found:
    raise FileNotFoundError(f"Không tìm thấy thư mục 'captions/' nào trong {DATA_ROOT}. "
                            f"Kiểm tra tên file dữ liệu trong DATASET_SOURCES (ô 0.2).")
for d in found:
    counts = []
    for split in ("train", "val", "test"):
        f = d / "captions" / f"{split}.csv"
        counts.append(len(pd.read_csv(f)) if f.exists() else None)
    key = next((k for k in EXPECTED_SPLIT if k in d.name.upper().replace("-", "")), None)
    exp = EXPECTED_SPLIT.get(key)
    mark = "✅" if exp and tuple(counts) == exp else "⚠️"
    print(f"{mark} {d.name}: train/val/test = {counts}" + (f"  (Bảng 2 MFGF: {exp})" if exp else ""))

### Train TVPReid-iLIDs: seed 1234 và 2025
Mỗi lần train xong sẽ tự mã hóa val/test và thu gọn checkpoint.

In [ ]:
rr("train", "--subs TVPReid-iLIDs", seeds_to_train("TVPReid-iLIDs"), "--no_ablations")

### Đánh giá các seed vừa train (chọn trọng số trên validation, đánh giá test một lần)

In [ ]:
rr("encode", "--subs TVPReid-iLIDs", "--seeds 42 1234 2025", "--no_ablations")
rr("eval", "--subs TVPReid-iLIDs", "--seeds 42 1234 2025", "--no_ablations")
for s in (42, 1234, 2025):
    f = Path(SRC_DIR, "results", "runs", "TVPReid-iLIDs", f"seed{s}.json")
    if f.exists():
        m = json.loads(f.read_text(encoding="utf-8"))["modes"]["val_selected"]["test"]["mfgf"]["metrics"]
        print(f"seed {s}: R@1 test = {m['R@1']:.2f}")
    else:
        print(f"seed {s}: chưa có")